# Online Retail Sales & Customer Value Analytics
**Online Retail Dataset | Dec 2010 – Dec 2011**

---

## Project Overview
End-to-end data analysis of a real-world transactional dataset from a UK-based online giftware retailer. The analysis covers data loading, cleaning, exploration, and statistical EDA. Business visualizations and dashboard reporting were produced in Power BI using the cleaned outputs from this notebook.

## Business Questions Answered
| # | Question |
|---|---|
| 1 | What is the gross revenue, net product revenue and revenue lost to cancellations/returns? |
| 2 | How does revenue change over time (monthly trends)? |
| 3 | Which countries generate the highest total revenue? |
| 4 | Who are the top 10 highest revenue-generating customers? |
| 5 | What is the split between repeat vs. one-time buyers? |
| 6 | Which products generate the highest revenue? |
| 7 | What is the proportion and financial impact of returns/cancellations? |
| 8 | What is the average value of a single customer order, and how does it vary by country (AOV)? |

## Tools Used
- **Python** (pandas, numpy) — data cleaning and EDA
- **Power BI** — interactive dashboard and business visualisations

## Dataset Source
Vijay UV (2019). *Online Retail* [Dataset]. Kaggle.
https://www.kaggle.com/datasets/vijayuv/onlineretail/data 


## Section 1: Import Libraries

The following libraries are imported for data manipulation and analysis:

- **pandas** — data loading, cleaning, transformation and aggregation
- **numpy** — numerical operations and array handling

In [45]:
import pandas as pd
import numpy as np

## Section 2: Load Dataset

The dataset is downloaded from a local CSV file sourced from Kaggle 
(https://www.kaggle.com/datasets/vijayuv/onlineretail/data) and loaded.

The `ISO-8859-1` encoding is specified because the file contains special 
characters in product descriptions (e.g. accented letters) that would cause 
a standard UTF-8 parser to fail.

The raw row count is printed immediately after loading to establish a 
baseline before any cleaning is applied.

In [38]:
# 1. Load the dataset
print ("Loading dataset....")
df = pd.read_csv (r"C:\Users\USER\Downloads\OnlineRetail.csv", encoding='ISO-8859-1')

# Capture state BEFORE cleaning
before_rows  = len(df)
before_nulls = df.isnull().sum()
print (f"Raw Row Count: {before_rows:,}")

Loading dataset....
Raw Row Count: 541,909


## Section 3: Initial Exploration

Before any cleaning is applied, the dataset is inspected across four dimensions:

| Check | Purpose |
|---|---|
| `df.info()` | Reveals column data types and confirms which columns have missing values |
| `df.describe()` | Provides summary statistics (mean, median, min, max, std) for numeric columns |
| `df.isnull().sum()` | Quantifies the exact number of missing values per column |
| `df.duplicated().sum()` | Counts exact duplicate rows that would inflate counts and revenue if retained |

**Key observations from this step:**
- `CustomerID` has 135,080 missing values — these represent guest transactions 
  and will be retained rather than dropped to avoid undercounting transaction volume
- `Description` has some blank/NaN values — will be replaced with 'UNKNOWN ITEMS'
- `InvoiceDate` is stored as a string — needs converting to datetime for trend analysis
- Duplicate rows exist and will be removed in the cleaning step

In [39]:
# Capture baseline metrics BEFORE cleaning for comparison
before_rows        = len(df)
before_nulls_cust  = df['CustomerID'].isnull().sum()
before_nulls_desc  = df['Description'].isnull().sum()
before_nulls_date  = df['InvoiceDate'].isnull().sum()
before_duplicates  = df.duplicated().sum()

print()
print("================= BASELINE SNAPSHOT (Before Cleaning) =================")
print(f"Total Rows:              {before_rows:,}")
print(f"CustomerID nulls:        {before_nulls_cust:,}")
print(f"Description nulls:       {before_nulls_desc:,}")
print(f"InvoiceDate nulls:       {before_nulls_date:,}")
print(f"Duplicate Rows:          {before_duplicates:,}")


================= BASELINE SNAPSHOT (Before Cleaning) =================
Total Rows:              541,909
CustomerID nulls:        135,080
Description nulls:       1,454
InvoiceDate nulls:       0
Duplicate Rows:          5,268


## Section 4: Data Cleaning

The following cleaning steps are applied sequentially to prepare the 
dataset for analysis. Each decision is documented with its justification.

| Step | Action | Justification |
|---|---|---|
| 1 | Strip whitespace & force uppercase on text columns (InvoiceNo, StockCode, Description, Country) | Prevents mismatches caused by hidden spaces or inconsistent casing |
| 2a | Replace blank/NaN descriptions with 'UNKNOWN ITEMS' | Retains rows while clearly flagging incomplete records |
| 2b | Fill missing CustomerID with 0 | Missing CustomerIDs represent guest transactions — removing them would undercount actual transaction volume |
| 3 | Convert InvoiceDate to datetime | Enables time-based analysis and date-level filtering |
| 4 | Create YearMonth column | Provides a clean period field for chronological trend plotting |
| 5 | Enforce correct data types on UnitPrice (float) and Quantity (int) | Prevents type-related errors in revenue calculations |
| 6 | Remove duplicate rows | Eliminates exact duplicates that would inflate transaction counts and revenue |
| 7 | Create TransactionID surrogate key | Assigns a unique identifier to every row for traceability |
| 8 | Calculate Revenue column (Quantity × UnitPrice) | Creates the core derived metric used across all revenue analysis |
| 9 | Export cleaned dataset to CSV | Produces a clean file ready for Power BI import |

In [40]:
### DATA CLEANING
# 1. Change the datatype for the columns, strip hidden spaces and force upper case
text_columns = ["InvoiceNo", "StockCode", "Description", "Country"]
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

df["StockCode"] = df["StockCode"].str.upper()

# 2a. Handle Missing/Blank Values
df["Description"] = df["Description"].replace(["nan",""],"UNKOWN ITEMS")

# 2b. Fill missing CustomerID with 0
df["CustomerID"] = df["CustomerID"].fillna(0)

# 3. Chnage the type of InvoiceDate column to DateTime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format = "mixed", errors = "coerce")

# 4. Create a clean 'Year-Month' column for chronological plotting (e.g., '2025-01')
df.insert(5, 'YearMonth', df['InvoiceDate'].dt.to_period('M'))

# 5. Change the datatype of the quantitative columns
df["UnitPrice"] = df["UnitPrice"].astype(float)

df["Quantity"] = df["Quantity"].astype(int)

# 6. Remove Duplicates
df = df.drop_duplicates(keep = "first")

# 7. Create a surrogate primary key
df.insert(0, "TransactionID", range(1, len(df)+1))

# 8. Calculate Revenue first on the master dataframe
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# 9. Save the clean dataset
df.to_csv(r"C:\Users\USER\Desktop\OnlineRetail_Cleaned_In_Python.csv", index = False)
print("The cleaned dataset is saved on your Desktop folder")
print()
print("================= BEFORE vs AFTER CLEANING =================")
print(f"{'Metric':<25} {'Before':>10} {'After':>10}")
print("-" * 47)
print(f"{'Row Count':<25} {before_rows:>10,} {len(df):>10,}")
print(f"{'CustomerID nulls':<25} {before_nulls_cust:>10,} {df['CustomerID'].isnull().sum():>10,}")
print(f"{'Description nulls':<25} {before_nulls_desc:>10,} {df['Description'].isnull().sum():>10,}")
print(f"{'InvoiceDate nulls':<25} {before_nulls_date:>10,} {df['InvoiceDate'].isnull().sum():>10,}")
print(f"{'Duplicate Rows':<25} {before_duplicates:>10,} {df.duplicated().sum():>10,}")

The cleaned dataset is saved on your Desktop folder

================= BEFORE vs AFTER CLEANING =================
Metric                        Before      After
-----------------------------------------------
Row Count                    541,909    536,641
CustomerID nulls             135,080          0
Description nulls              1,454          0
InvoiceDate nulls                  0          0
Duplicate Rows                 5,268          0


## Section 5: DataFrame Branching

The cleaned master dataset is split into two focused analytical pipelines 
to separate fundamentally different types of transactions.

| DataFrame | Filter Conditions | Purpose |
|---|---|---|
| `sales_df` | `UnitPrice > 0`, `Quantity > 0`, InvoiceNo not starting with 'A' | Core commercial sales — money exchanged for goods |
| `returns_df` | `UnitPrice > 0`, `Quantity < 0` | Customer returns — money refunded |

> **Audit Note — 'A' prefix invoices excluded:**
> InvoiceNo entries prefixed with 'A' were identified as manual double-entry 
> bookkeeping corrections (e.g. a large 'Adjust bad debt' entry at 14:50 
> immediately negated by a reversal at 14:51). Retaining these would distort 
> gross revenue and transaction counts. They are excluded from `sales_df` but 
> remain in the master dataset for audit traceability.

Both `sales_df` and `returns_df` were exported alongside the master `df` 
for use in Power BI dashboard visualisations.

In [41]:
# 1. Calculate Revenue first on the master dataframe
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# 2. Standard Purchase Transactions
sales_df = df[(df['UnitPrice'] > 0) & (df['Quantity'] > 0) & (~df['InvoiceNo'].astype(str).str.startswith('A'))].copy()

# BRANCH 2: THE CUSTOMER RETURNS PIPELINE
# Captures official customer returns where money was refunded
returns_df = df[(df['UnitPrice'] > 0) & (df['Quantity'] < 0)].copy()

# VERIFICATION CHECK
print("=== DATAFRAME BRANCHING COMPLETE ===")
print(f"Master Dataset Total Rows:   {len(df):,}")
print(f"1. Core Sales Rows:          {len(sales_df):,}")
print(f"2. Customer Return Rows:     {len(returns_df):,}")

=== DATAFRAME BRANCHING COMPLETE ===
Master Dataset Total Rows:   536,641
1. Core Sales Rows:          524,877
2. Customer Return Rows:     9,251


## Section 6: Statistical EDA

Descriptive statistics and correlation analysis are computed on the 
cleaned master dataset to understand revenue distribution and relationships 
between key numeric variables.

**What to look for:**

- **Mean vs Median Revenue:** A large gap between the two indicates right skewness, 
  expected here given wholesale bulk orders inflating the upper end of the distribution
- **Standard Deviation:** A high std relative to the mean signals high variability 
  in transaction sizes — consistent with a mix of retail and wholesale customers
- **Quantity vs Revenue Correlation:** Expected to be strongly positive — 
  more units generally means higher revenue
- **UnitPrice vs Quantity Correlation:** A negative value here would suggest 
  higher-priced items are bought in smaller quantities, consistent with 
  wholesale purchasing behaviour

In [42]:
# STATISTICAL SUMMARY
print("=== STATISTICAL SUMMARY ===")
print(f"Mean Revenue per Transaction: £{df['Revenue'].mean():,.2f}")
print(f"Median Revenue per Transaction: £{df['Revenue'].median():,.2f}")
print(f"Std Deviation: £{df['Revenue'].std():,.2f}")
print(f"Revenue Range: £{df['Revenue'].min():,.2f} to £{df['Revenue'].max():,.2f}")
print("------------------------------------------")
print(f"Revenue Distribution:")
print(df['Revenue'].describe().to_frame().map('{:,.2f}'.format))
print("------------------------------------------")

# RELATIONSHIP ANALYSIS (Correlation)
print("\n=== KEY RELATIONSHIPS ===")
print(f"Correlation (Quantity vs Revenue): {df['Quantity'].corr(df['Revenue']):.3f}")
print(f"Correlation (UnitPrice vs Quantity): {df['UnitPrice'].corr(df['Quantity']):.3f}")

=== STATISTICAL SUMMARY ===
Mean Revenue per Transaction: £18.12
Median Revenue per Transaction: £9.87
Std Deviation: £380.66
Revenue Range: £-168,469.60 to £168,469.60
------------------------------------------
Revenue Distribution:
           Revenue
count   536,641.00
mean         18.12
std         380.66
min    -168,469.60
25%           3.75
50%           9.87
75%          17.40
max     168,469.60
------------------------------------------

=== KEY RELATIONSHIPS ===
Correlation (Quantity vs Revenue): 0.887
Correlation (UnitPrice vs Quantity): -0.001


## Section 7: Returns Analysis

A product-level breakdown of returns is computed from `returns_df` to 
identify which items are most frequently returned and how much revenue 
is lost to each.

**Key finding:** Paper Craft Little Birdie appears as the 2nd highest 
revenue product in the sales pipeline but was returned in full by a 
single customer (CustomerID 16446) in December 2011. Its true net 
revenue contribution is effectively £0 — making its high ranking 
entirely misleading for procurement and stock planning purposes.

In [46]:
# List of non-product operational StockCodes to exclude
exclude_codes = ["C2", "AMAZONFEE", "M", "BANK CHARGES", "POST", "D", "S", "DOT", "CRUK"]

# Filter out the operational descriptions using the ~ (NOT) operator
cleaned_returns_df = returns_df[~returns_df['StockCode'].isin(exclude_codes)]

# Calculate top 10 most returned actual products by revenue lost
top_returns = cleaned_returns_df.groupby('Description')['Revenue'].sum().abs()
top_returns = top_returns.sort_values(ascending=False).head(10)

print("=== TOP 10 MOST RETURNED PRODUCTS ===")
print(top_returns.map('£{:,.2f}'.format))

=== TOP 10 MOST RETURNED PRODUCTS ===
Description
PAPER CRAFT , LITTLE BIRDIE           £168,469.60
MEDIUM CERAMIC TOP STORAGE JAR         £77,479.64
REGENCY CAKESTAND 3 TIER                £9,697.05
WHITE HANGING HEART T-LIGHT HOLDER      £6,624.30
FAIRY CAKE FLANNEL ASSORTED COLOUR      £6,591.42
PANTRY CHOPPING BOARD                   £4,803.06
DOORMAT FAIRY CAKE                      £4,554.90
GIN + TONIC DIET METAL SIGN             £3,775.33
TEA TIME PARTY BUNTING                  £3,692.95
FELTCRAFT DOLL MOLLY                    £3,512.65
Name: Revenue, dtype: object


## Section 8: Key Insights & Recommendations

The following insights were derived from the statistical EDA above
and the Power BI dashboard. Each is supported by data evidence
and paired with a business recommendation.

| # | Insight | Recommendation |
|---|---|---|
| 1 | Revenue health is strong: Returns rate is just 4.64%, retaining 95.36% of gross revenue (£9.77M of £10.25M) | Implement mandatory return-reason capture at point of refund to identify and eliminate recurring return drivers |
| 2 | Strong Q4 seasonality: Revenue peaks at £1.50M in November driven by Christmas gift purchasing; December 2011 drop is not a true decline as the dataset only covers the first 9 days | Increase stock levels and marketing spend from September onwards to fully capitalise on Q4 demand |
| 3 | International markets split into two profiles: High-volume (Netherlands £285.45K, EIRE £283.14K) and high-value (Netherlands £3.05K AOV, Australia £2.45K AOV); Singapore and Japan show high AOV but low revenue, signalling untapped growth potential | Retention and account management focus for high-AOV markets; customer acquisition focus for Singapore and Japan |
| 4 | Revenue dangerously concentrated: Top 2 customers contribute 5.53% of net revenue combined; 34.41% of customers are one-time buyers who must be actively retained as purchases are discretionary, not necessity-driven | VIP account management and loyalty incentives for top customers; post-purchase surveys and tiered loyalty programme to convert one-time buyers |
| 5 | Top products are decorative seasonal gift-ware consistent with Q4 demand; Paper Craft Little Birdie (ranked 2nd at £168.47K) is entirely driven by a single bulk order from CustomerID 16446 that was subsequently returned in full — true net revenue is effectively £0 | Procurement must validate stock decisions against Top Products by Unique Customers metric, never revenue alone; formalise Wholesale vs Retail customer segmentation with appropriate pricing policies |

> **Full insight report available in:** `Online_Retail_Insight_Report.docx`